# TSFM RUL playbook — Phase B: the three REAL industrial datasets

**Run all** clones the code from GitHub and runs the Phase-B campaign: **MetroPT-3**
(real metro-train air compressor, CENSORED), **UCI Hydraulic** (real rig, the RQ-F
adjustment-vs-replacement anchor) and **Backblaze Drive Stats** (real drive fleet,
CENSORED at scale). Sibling of the per-family notebooks `cmapss.ipynb`, `xjtu.ipynb`,
`ncmapss.ipynb` — run it on its own Colab runtime, in parallel with them.

**These three datasets do NOT all produce a RUL curve, by design** (CHANGES.md §54–§56):

| dataset | what it is | what the campaign runs | headline output |
|---|---|---|---|
| **MetroPT-3** | mostly-healthy fleet, 4 documented air-leak events, right-censored tail | the binary **alarm** sweep | `<ds>_<model>_alarm_results.csv` + alarm-scaling figures |
| **Backblaze** | ~1-in-23,500 drive-days fail, most drives right-censored | the binary **alarm** sweep | `<ds>_<model>_alarm_results.csv` + alarm-scaling figures |
| **Hydraulic** | a fault-INJECTION rig with **no failure events at all** | the **RQ-F taxonomy probe** | `<ds>_<model>_taxonomy.csv` + the few-shot curve |

The alarm metrics (precision / recall / AUROC + lead time) share no scale with the RUL
ones, so they are written to a **separate CSV** and must never be tabled against
C-MAPSS's NASA scores. Hydraulic's RUL is degenerate by construction (uniform label
blocks give a constant target), so the campaign skips its RUL stages and the loader says
so out loud.

## 1. Setup — clone the code, mount Drive for data

In [ ]:
# Install the non-default deps (Colab already ships torch/numpy/pandas/sklearn/matplotlib).
%pip install -q "chronos-forecasting>=2.0.0" "coral-pytorch==1.4.0" "lightgbm>=4.0" "sktime>=1.0" "numba>=0.59" "safetensors>=0.4" "h5py>=3.10" "pycatch22>=0.4" "pyarrow>=14.0"

import os, sys, subprocess, torch
from google.colab import drive

# 1. Mount Drive -- it holds ONLY Data/, the embedding cache/, and results/ (what persists).
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Clone the CODE fresh from GitHub into Colab's ephemeral disk, so Drive never mirrors
#    the repo and you never re-upload changes. Re-run-safe: fast-forwards if already cloned.
REPO_URL    = 'https://github.com/blozanod/Predictive-Maintenance-LSTM.git'
REPO_BRANCH = 'main'                       # branch to run (a public repo needs no token)
CLONE_DIR   = '/content/Predictive-Maintenance-LSTM'
if not os.path.isdir(os.path.join(CLONE_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
                    REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)

# 3. Put the clone first on sys.path so `import src.*` resolves to the fresh code.
if CLONE_DIR in sys.path:
    sys.path.remove(CLONE_DIR)
sys.path.insert(0, CLONE_DIR)

print('repo :', CLONE_DIR, '@', REPO_BRANCH)
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Where to put the downloads

Each dataset goes in its own folder under one `Data/` root (`config.data_root`). Nothing
needs renaming — the loaders accept the shipped folder/file names and one nesting level
(CHANGES.md §26):

```
Data/
  MetroPT-3/   MetroPT3(AirCompressor).csv        # UCI 791 (MetroPT3.csv also accepted)
  Hydraulic/   PS1..PS6.txt EPS1.txt FS1..FS2.txt TS1..TS4.txt VS1.txt CE.txt CP.txt
               SE.txt profile.txt                # UCI 447 (18 tab-delimited files)
  Backblaze/   2024-01-01.csv 2024-01-02.csv …   # unzipped quarterly archives, any nesting
```

A dataset that is not there is **skipped with a notice**, so you can run this notebook
with only one of the three downloaded.

## 3. Config

`src/config.py` stays the single source of truth. The per-dataset protocol choices
(bin width, alarm horizon, window size, the drive-model scope) come from
`campaign.DEFAULT_DATASET_OVERRIDES` — inspect it below rather than hand-editing them.

In [ ]:
from src.config import Config

# Point this at YOUR Drive folder holding Data/ (the only thing that must live on Drive).
DRIVE = '/content/drive/MyDrive/Predictive Maintenance LSTM'
config = Config(
    data_root=f'{DRIVE}/Data',
    cache_dir=f'{DRIVE}/cache',
    results_dir=f'{DRIVE}/results',
    # The recorded §12 winner shape. Per-dataset overrides adjust window_size /
    # tsfm_context_length / max_rul / alarm_horizon where the units differ (hours for
    # MetroPT, 60 s rig cycles for Hydraulic, days for Backblaze).
    tsfm_context_length=256,
    head_features='emb+locscale',
    pooling='mean',
)
config

## 4. Inspect the recorded per-dataset protocol

Read this before you read any number: it states, per dataset, what a "cycle" and a "unit"
mean, and what the alarm horizon is measured in.

In [ ]:
from src.campaign import DEFAULT_DATASET_OVERRIDES, RUL_ONLY_STAGES, TIME_TO_EVENT_STAGES

for ds in ('MetroPT-3', 'Hydraulic', 'Backblaze'):
    cfg = config.replace(dataset=ds, sensor_columns=None, **DEFAULT_DATASET_OVERRIDES[ds])
    print(f'{ds:12s} kind={cfg.dataset_kind():10s} '
          f'censored={cfg.is_censored_dataset()!s:5s} '
          f'classification={cfg.is_classification_dataset()!s:5s} '
          f'channels={cfg.num_channels():3d} alarm_horizon={cfg.alarm_horizon}')
    print(f'{"":14s}overrides: {DEFAULT_DATASET_OVERRIDES[ds]}')

print()
print('RUL-only stages skipped on a censored fleet     :', RUL_ONLY_STAGES)
print('time-to-event stages skipped on Hydraulic       :', TIME_TO_EVENT_STAGES)

## 5. Campaign — the three Phase-B datasets

`run_campaign` routes each dataset to its own arm automatically
(`config.is_censored_dataset()` / `is_classification_dataset()`), and every stage is
restartable, so re-running resumes rather than recomputing. Datasets missing from `Data/`
are skipped with a notice.

In [ ]:
import torch
from src.campaign import run_campaign

device = 'cuda' if torch.cuda.is_available() else 'cpu'

summary = run_campaign(
    config,
    datasets=['MetroPT-3', 'Hydraulic', 'Backblaze'],
    models=['amazon/chronos-2'],          # add the other backbones on their own runtimes
    stages=['cache', 'sweep', 'fairness', 'horizon', 'figures'],
    device=device,
)
for row in summary:
    print(row['status'], row['dataset'], sorted(k for k in row if k.endswith('_csv')))

## 6. Score the censored chapter (the alarm / lead-time metric)

The win-rule is direction-aware: the alarm metrics are SKILL scores (higher is better),
unlike every RUL error metric, and `alarm_base_rate` is the floor that makes a hollow win
detectable (CHANGES.md §54e).

In [ ]:
from pathlib import Path
from src import scoring, plots
from src.evaluate import load_results

results_dir = Path(config.results_dir)
alarm_csvs = sorted(results_dir.glob('*alarm_results.csv'))
print('alarm result files:', [p.name for p in alarm_csvs])

for metric in ('alarm_ap', 'alarm_auroc'):
    if not alarm_csvs:
        print('no alarm results yet — download MetroPT-3 and/or Backblaze first')
        break
    table = scoring.success_map(str(results_dir / '*alarm_results.csv'), config,
                                metric=metric, secondary_metric='alarm_recall')
    print(f'\n=== {metric} ===')
    for row in table:
        print(f"  {row['dataset']:12s} n={row['n_units']:>5} {row['model']:22s} "
              f"{row['verdict']:6s} margin={row['margin']:+.3f} p={row['p']:.3f} "
              f"vs {row['best_baseline']}")

for path in alarm_csvs:
    plots.plot_alarm_scaling(path, config.figures_dir(),
                             prefix=Path(path).stem.replace('alarm_results', ''),
                             show=True)

## 7. Score the RQ-F chapter (adjustment vs. replacement)

The deliverable is the **gap between the embedding curve and the catch22 curve**: does a
frozen TSFM embedding separate a fault that needs an *adjustment* from one that needs a
*replacement*, with few labels, better than a hand-crafted indicator bank?

In [ ]:
taxonomy_csvs = sorted(results_dir.glob('*taxonomy.csv'))
print('taxonomy result files:', [p.name for p in taxonomy_csvs])

for path in taxonomy_csvs:
    rows = load_results(path)
    by = {}
    for r in rows:
        by.setdefault((r['feature_source'], int(r['shots'])), []).append(float(r['macro_f1']))
    print(f'\n=== {path.name} (macro-F1, seed-mean) ===')
    for (source, shots) in sorted(by):
        vals = by[(source, shots)]
        print(f'  {source:14s} shots={shots:>4}  {sum(vals)/len(vals):.3f}  (n={len(vals)})')
    plots.plot_taxonomy(path, config.figures_dir(),
                        prefix=Path(path).stem.replace('taxonomy', ''), show=True)

## 8. Optional — the Phase-B factor probes

RQ-D (raw vs. indicators, on XJTU's 25.6 kHz waveforms) and RQ-G (sampling rate and
aggregation, on N-CMAPSS's 1 Hz within-flight rows). Both need their own datasets
downloaded; set `RUN_PROBES = True` to run them.

In [ ]:
RUN_PROBES = False

if RUN_PROBES:
    from src.probes import run_factor_probe

    # RQ-D -- do TSFMs make hand-crafted condition indicators obsolete?
    xjtu = config.replace(dataset='XJTU-SY', sensor_columns=None,
                          **DEFAULT_DATASET_OVERRIDES['XJTU-SY'])
    run_factor_probe(xjtu, 'feature_mode', levels={
        'indicators':  {'xjtu_feature_mode': 'indicators'},
        'raw_decimate': {'xjtu_feature_mode': 'raw', 'xjtu_raw_reduce': 'decimate'},
        'raw_segrms':  {'xjtu_feature_mode': 'raw', 'xjtu_raw_reduce': 'segment_rms'},
        'raw_plus':    {'xjtu_feature_mode': 'raw+indicators'},
    }, models=['amazon/chronos-2'], baselines=['gbm', 'catch22_gbm', 'predict_mean'],
       device=device)

    # RQ-G -- how finely must you sample, and how should sub-cycle data be aggregated?
    ds02 = config.replace(dataset='DS02', sensor_columns=None)
    run_factor_probe(ds02, 'aggregation', levels={
        '1hz_meanstd': {'ncmapss_agg_stride': 1,  'ncmapss_agg_stats': 'mean_std'},
        'stride10':    {'ncmapss_agg_stride': 10, 'ncmapss_agg_stats': 'mean_std'},
        'stride60':    {'ncmapss_agg_stride': 60, 'ncmapss_agg_stats': 'mean_std'},
        'rich':        {'ncmapss_agg_stride': 1,  'ncmapss_agg_stats': 'mean_std_minmax_slope'},
    }, models=['amazon/chronos-2'], baselines=['gbm', 'predict_mean'], device=device)
else:
    print('set RUN_PROBES = True to run the RQ-D / RQ-G factor probes')